### Wörter aufräumen – Lösung Schritt für Schritt

*39. Bundeswettbewerb Informatik (2020), 1. Runde, Aufgabe 1*

Opa Jürgen blättert in einer Zeitschrift aus der Apotheke und findet ein Rätsel. Es ist eine Liste von Wörtern gegeben, die in die richtige Reihenfolge gebracht werden sollen,
so dass sie eine lustige Geschichte ergeben. Leerzeichen und Satzzeichen sowie einige Buchstaben sind schon vorgegeben.

„Oh je, was für eine Arbeit“, sagt Opa Jürgen. „Kein Problem“ erwidert seine Enkelin Lotta. „Der Computer kann es doch für uns machen.“

#### Aufgabe 1

Schreibe ein Programm, das einen Lückentext und eine Liste von Wörtern einliest und anschließend den vervollständigten Text ausgibt.
Du kannst davon ausgehen, dass es nur eine richtige Lösung gibt. Wende dein Programm auf die Beispiele (`raetsel0.txt` bis `raetsel4.txt`) an.

#### Beispieldaten

- 1. Zeile: der Lückentext (`_` = fehlender Buchstabe), 2. Zeile: alle Wörter, durch Leerzeichen getrennt

`raetsel0.txt`:

```
_h __, _a_ __r ___e __b___!
arbeit eine für je oh was
```

## Was dir bei dieser Aufgabe hilft

**Algorithmische Kenntnisse**
- **Zuordnungsproblem (Matching):** Jede Lücke bekommt genau ein Wort, jedes Wort wird genau einmal benutzt. Man modelliert das als Graph mit Lücken und Wörtern als Knoten und einer Kante für „passt“.
- **Backtracking:** Eine Lücke nach der anderen füllen, bei einer Sackgasse den letzten Schritt zurücknehmen und ein anderes Wort probieren.
- **Verbessernde Wege:** Das schnellere Verfahren (für Neugierige): Ist das passende Wort schon vergeben, versucht man, dem bisherigen Besitzer ein anderes Wort zu geben.
- **Früh aussortieren:** Passt die Länge nicht, muss man die Buchstaben gar nicht erst vergleichen.

**Nützliche Funktionen**
- `text.split()` trennt an Leerzeichen, `' '.join(liste)` setzt wieder zusammen.
- `wort.isalpha()` prüft, ob ein Zeichen ein Buchstabe ist (auch ä, ö, ü, ß). Damit erkennt man Satzzeichen am Ende einer Lücke.
- `open(datei, encoding='utf-8-sig')` liest Umlaute richtig.

## Die Idee

Drei Teilschritte:

1. **Lücken erkennen:** Der Lückentext wird an den Leerzeichen getrennt. Jedes Stück ist eine Lücke, zum Beispiel `___e` oder `__b___!`.
   Ein **Satzzeichen am Ende** (`!`, `.`, `,`) gehört nicht zum Wort. Wir schneiden es ab, merken es uns und hängen es bei der Ausgabe wieder an.
2. **Passende Wörter finden:** Ein Wort passt zu einer Lücke, wenn es gleich lang ist und an jeder Stelle, an der die Lücke einen Buchstaben vorgibt, derselbe Buchstabe steht (`_` passt zu jedem).
   Für jede Lücke legen wir die Liste der passenden Wörter an (als Positionen in der Wortliste).
3. **Zuordnung finden:** Passen zu einer Lücke mehrere Wörter (im Beispiel passen zu `__,` die Wörter `je` und `oh`), dürfen wir nicht einfach das erste nehmen:
   Dann fehlt es womöglich einer anderen Lücke (`_h` braucht `oh`). Wir probieren per **Backtracking**.

Zwei Wörter können gleich sein (in `raetsel1.txt` kommt `wurde` zweimal vor). Weil wir Wörter über ihre **Position** in der Liste führen, ist das kein Problem.

## Schritt 1: Einlesen

Die erste Zeile ist der Lückentext, die zweite die Wortliste. Wir geben die Lücken und die Wörter als Listen zurück.

In [ ]:
def lies_raetsel(dateiname):
    with open(dateiname, encoding='utf-8-sig') as f:
        zeilen = f.read().splitlines()
    luecken = zeilen[0].split()
    woerter = zeilen[1].split()
    return luecken, woerter


luecken, woerter = lies_raetsel('beispieldaten/raetsel0.txt')
print(luecken)
print(woerter)

['_h', '__,', '_a_', '__r', '___e', '__b___!']
['arbeit', 'eine', 'für', 'je', 'oh', 'was']


## Schritt 2: Satzzeichen abtrennen und Wörter vergleichen

`zerlege(luecke)` liefert das **Muster** (der Anfang der Lücke ohne Satzzeichen am Ende) und das **Satzzeichen-Ende** als Text. Wir gehen vom Ende her, solange das Zeichen weder `_` noch ein Buchstabe ist.
Bei `__b___!` ist das Muster `__b___` und das Ende `!`.

In [ ]:
def zerlege(luecke):
    ende = len(luecke)
    while ende > 0 and luecke[ende - 1] != '_' and not luecke[ende - 1].isalpha():
        ende = ende - 1
    return luecke[:ende], luecke[ende:]


def passt(muster, wort):
    if len(muster) != len(wort):                  # Länge zuerst: schnellster Test
        return False
    for i in range(len(wort)):
        if muster[i] != '_' and muster[i] != wort[i]:
            return False
    return True


print(zerlege('__b___!'), zerlege('__,'), zerlege('_h'))
print(passt('__b___', 'arbeit'), passt('__b___', 'eine'), passt('___e', 'eine'), passt('___e', 'arbeit'))

('__b___', '!') ('__', ',') ('_h', '')
True False True False


## Schritt 3: Kandidaten je Lücke

Für jede Lücke sammeln wir die Positionen aller passenden Wörter. Das ist der Graph „Lücke – Wort“ als Liste von Listen.

In [ ]:
def finde_kandidaten(muster_liste, woerter):
    kandidaten = []
    for muster in muster_liste:
        liste = []
        for j in range(len(woerter)):
            if passt(muster, woerter[j]):
                liste.append(j)
        kandidaten.append(liste)
    return kandidaten


muster_liste = [zerlege(l)[0] for l in luecken]
kandidaten = finde_kandidaten(muster_liste, woerter)
for i in range(len(luecken)):
    print(luecken[i], '->', [woerter[j] for j in kandidaten[i]])

_h -> ['oh']
__, -> ['je', 'oh']
_a_ -> ['was']
__r -> ['für']
___e -> ['eine']
__b___! -> ['arbeit']


Hier sieht man schon, warum „das erste passende Wort nehmen“ nicht reicht: Die Lücke `__,` hat `je` und `oh` als Kandidaten, aber `oh` wird für `_h` gebraucht.
Wie sich blindes Nehmen des ersten freien Wortes verhält, probieren wir im nächsten Schritt aus.

## Schritt 4: Das erste freie Wort nehmen (geht schief)

Zum Vergleich der einfache Ansatz: Gehe die Lücken der Reihe nach durch und nimm das erste passende Wort, das noch frei ist. Das nennt man ein **Greedy-Verfahren** („gierig“).
`verwendet[j]` merkt sich, ob Wort `j` schon vergeben ist.

In [ ]:
def greedy(kandidaten, anzahl_woerter):
    verwendet = [False] * anzahl_woerter
    zuordnung = []
    for liste in kandidaten:
        gewaehlt = None
        for j in liste:
            if not verwendet[j]:
                gewaehlt = j
                break
        if gewaehlt is None:
            return None                          # Sackgasse
        verwendet[gewaehlt] = True
        zuordnung.append(gewaehlt)
    return zuordnung


# kleines eigenes Beispiel: zwei Lücken, zwei Wörter
print(greedy(finde_kandidaten(['__', '_b'], ['ab', 'cd']), 2))

None


Die erste Lücke `__` nimmt `ab` weg, obwohl `_b` dieses Wort braucht. Dabei gibt es eine Lösung: `__` bekommt `cd`, `_b` bekommt `ab`.
Das Greedy-Verfahren ist also falsch (die Bewertungskriterien sagen das ausdrücklich). Wir brauchen ein Verfahren, das einen Schritt **zurücknehmen** kann.

## Schritt 5: Backtracking

`fuelle(i, ...)` füllt die Lücken ab Nummer `i`. Für Lücke `i` probiert die Funktion der Reihe nach alle Kandidaten, die noch frei sind:
Wort als verwendet markieren, die nächste Lücke füllen (rekursiv). Klappt das nicht, wird die Markierung zurückgenommen und das nächste Kandidatenwort probiert.
Ist `i` gleich der Anzahl der Lücken, sind alle gefüllt, und die Zuordnung ist fertig.

Die Liste `zaehler` zählt, wie oft die Funktion aufgerufen wird (so können wir Verfahren vergleichen, ohne Zeiten zu messen).

In [ ]:
def fuelle(i, kandidaten, zuordnung, verwendet, zaehler):
    zaehler[0] = zaehler[0] + 1
    if i == len(kandidaten):
        return True                              # alle Lücken gefüllt
    for j in kandidaten[i]:
        if not verwendet[j]:
            verwendet[j] = True
            zuordnung[i] = j
            if fuelle(i + 1, kandidaten, zuordnung, verwendet, zaehler):
                return True
            verwendet[j] = False                 # zurücknehmen und weiter probieren
    return False


def finde_zuordnung(kandidaten, anzahl_woerter):
    zuordnung = [None] * len(kandidaten)
    verwendet = [False] * anzahl_woerter
    zaehler = [0]
    if fuelle(0, kandidaten, zuordnung, verwendet, zaehler):
        return zuordnung, zaehler[0]
    return None, zaehler[0]


print(finde_zuordnung(finde_kandidaten(['__', '_b'], ['ab', 'cd']), 2))

([1, 0], 4)


Die Lücke `__` bekommt Wort 1 (`cd`), die Lücke `_b` Wort 0 (`ab`). Vier Aufrufe für zwei Lücken: Der erste Versuch (`__` mit `ab`) läuft bei `_b` in eine Sackgasse und wird zurückgenommen, danach klappt `__` mit `cd`, `_b` mit `ab`, und ein letzter Aufruf stellt fest, dass alle Lücken gefüllt sind.

## Schritt 6: Alles zusammen, mit Ausgabe

Aus der Zuordnung wird der fertige Text: Jede Lücke wird durch ihr Wort ersetzt, danach kommt das gemerkte Satzzeichen-Ende.

In [ ]:
def loese(dateiname):
    luecken, woerter = lies_raetsel(dateiname)
    muster_liste = []
    enden = []
    for luecke in luecken:
        muster, ende = zerlege(luecke)
        muster_liste.append(muster)
        enden.append(ende)
    kandidaten = finde_kandidaten(muster_liste, woerter)
    zuordnung, aufrufe = finde_zuordnung(kandidaten, len(woerter))
    if zuordnung is None:
        return None, aufrufe
    teile = []
    for i in range(len(luecken)):
        teile.append(woerter[zuordnung[i]] + enden[i])
    return ' '.join(teile), aufrufe


for nr in range(5):
    text, aufrufe = loese(f'beispieldaten/raetsel{nr}.txt')
    print(f'raetsel{nr} ({aufrufe} Aufrufe):')
    print(text)
    print()

raetsel0 (7 Aufrufe):
oh je, was für eine arbeit!

raetsel1 (65 Aufrufe):
Am Anfang wurde das Universum erschaffen. Das machte viele Leute sehr wütend und wurde allenthalben als Schritt in die falsche Richtung angesehen.

raetsel2 (34 Aufrufe):
Als Gregor Samsa eines Morgens aus unruhigen Träumen erwachte, fand er sich in seinem Bett zu einem ungeheueren Ungeziefer verwandelt.

raetsel3 (177 Aufrufe):
Informatik ist die Wissenschaft von der systematischen Darstellung, Speicherung, Verarbeitung und Übertragung von Informationen, besonders der automatischen Verarbeitung mit Digitalrechnern.

raetsel4 (22182 Aufrufe):
Opa Jürgen blättert in einer Zeitschrift aus der Apotheke und findet ein Rätsel. Es ist eine Liste von Wörtern gegeben, die in die richtige Reihenfolge gebracht werden sollen, so dass sie eine lustige Geschichte ergeben. Leerzeichen und Satzzeichen sowie einige Buchstaben sind schon vorgegeben.



Alle fünf Rätsel sind gelöst, die Texte stimmen mit den Lösungen der Lösungshinweise überein. Beachte die Aufrufe: Bei `raetsel4.txt` (44 Lücken) braucht das Backtracking in dieser Reihenfolge
über 22 000 Aufrufe, obwohl 45 genügen würden (eine pro Lücke plus Schluss). Es verrennt sich oft und muss zurück.

## Schritt 7: Schwierige Lücken zuerst

Eine Lücke mit nur **einem** Kandidaten hat keine Wahl, ihr Wort ist festgelegt. Je früher man solche Lücken füllt, desto früher fehlt das Wort den anderen Lücken, und falsche Wege enden schneller.
Wir sortieren die Lücken deshalb nach der **Anzahl ihrer Kandidaten**, die wenigsten zuerst.

Sortieren von Paaren: `paare.sort()` sortiert eine Liste von Tupeln `(anzahl, i)` zuerst nach `anzahl`, bei Gleichstand nach `i`. Aus dem sortierten Paar liest man ab, welche Lücke an der Reihe ist.

In [ ]:
def finde_zuordnung_sortiert(kandidaten, anzahl_woerter):
    paare = []
    for i in range(len(kandidaten)):
        paare.append((len(kandidaten[i]), i))
    paare.sort()
    sortiert = []                                # Kandidatenlisten in der neuen Reihenfolge
    for anzahl, i in paare:
        sortiert.append(kandidaten[i])
    zuordnung_sortiert, aufrufe = finde_zuordnung(sortiert, anzahl_woerter)
    if zuordnung_sortiert is None:
        return None, aufrufe
    zuordnung = [None] * len(kandidaten)         # zurück auf die ursprüngliche Reihenfolge
    for platz in range(len(paare)):
        i = paare[platz][1]
        zuordnung[i] = zuordnung_sortiert[platz]
    return zuordnung, aufrufe


for nr in range(5):
    luecken, woerter = lies_raetsel(f'beispieldaten/raetsel{nr}.txt')
    kandidaten = finde_kandidaten([zerlege(l)[0] for l in luecken], woerter)
    _, auf1 = finde_zuordnung(kandidaten, len(woerter))
    _, auf2 = finde_zuordnung_sortiert(kandidaten, len(woerter))
    print(f'raetsel{nr}: {len(luecken)} Lücken, Aufrufe ohne Sortieren {auf1}, mit Sortieren {auf2}')

raetsel0: 6 Lücken, Aufrufe ohne Sortieren 7, mit Sortieren 7
raetsel1: 22 Lücken, Aufrufe ohne Sortieren 65, mit Sortieren 23
raetsel2: 20 Lücken, Aufrufe ohne Sortieren 34, mit Sortieren 21
raetsel3: 20 Lücken, Aufrufe ohne Sortieren 177, mit Sortieren 22
raetsel4: 44 Lücken, Aufrufe ohne Sortieren 22182, mit Sortieren 45


Mit der Sortierung genügt bei `raetsel4.txt` ein Aufruf pro Lücke (plus Schluss), das Backtracking muss gar nicht mehr zurück. Eine Garantie ist das nicht, aber für diese Rätsel reicht es.
Wer eine Garantie möchte, nimmt das Verfahren aus dem nächsten Abschnitt.

## Für Neugierige: Verbessernde Wege (bipartites Matching)

Die Lösungshinweise zeigen ein Verfahren, das immer schnell ist (Laufzeit höchstens Lücken mal Kanten, also kubisch). Die Idee: Wir füllen die Lücken **eine nach der anderen**.
Hat eine Lücke einen Kandidaten, der noch frei ist, nehmen wir ihn. Ist er schon vergeben, fragen wir den **bisherigen Besitzer**, ob er auf ein anderes Wort ausweichen kann (rekursiv, mit einem Zettel `besucht`, damit wir uns nicht im Kreis drehen).
Klappt das, tauschen wir. Diesen Weg nennt man *verbessernden Weg*: Am Ende ist ein Paar mehr gebildet. Anders als beim Backtracking wird nie etwas komplett verworfen.

In [ ]:
def verbessere(i, kandidaten, besitzer, besucht, zaehler):
    zaehler[0] = zaehler[0] + 1
    for j in kandidaten[i]:
        if not besucht[j]:
            besucht[j] = True
            if besitzer[j] is None or verbessere(besitzer[j], kandidaten, besitzer, besucht, zaehler):
                besitzer[j] = i                  # Wort j gehört jetzt der Lücke i
                return True
    return False


def finde_zuordnung_matching(kandidaten, anzahl_woerter):
    besitzer = [None] * anzahl_woerter           # besitzer[j]: Lücke, die Wort j hat
    zaehler = [0]
    for i in range(len(kandidaten)):
        besucht = [False] * anzahl_woerter
        if not verbessere(i, kandidaten, besitzer, besucht, zaehler):
            return None, zaehler[0]
    zuordnung = [None] * len(kandidaten)
    for j in range(anzahl_woerter):
        if besitzer[j] is not None:
            zuordnung[besitzer[j]] = j
    return zuordnung, zaehler[0]


for nr in range(5):
    luecken, woerter = lies_raetsel(f'beispieldaten/raetsel{nr}.txt')
    kandidaten = finde_kandidaten([zerlege(l)[0] for l in luecken], woerter)
    zuordnung, aufrufe = finde_zuordnung_matching(kandidaten, len(woerter))
    ok = all(passt(zerlege(luecken[i])[0], woerter[zuordnung[i]]) for i in range(len(luecken)))
    print(f'raetsel{nr}: {aufrufe} Aufrufe, jedes Wort einmal benutzt: {sorted(zuordnung) == list(range(len(woerter)))}, alle passen: {ok}')
print(greedy(finde_kandidaten(['__', '_b'], ['ab', 'cd']), 2), finde_zuordnung_matching(finde_kandidaten(['__', '_b'], ['ab', 'cd']), 2))

raetsel0: 6 Aufrufe, jedes Wort einmal benutzt: True, alle passen: True
raetsel1: 32 Aufrufe, jedes Wort einmal benutzt: True, alle passen: True
raetsel2: 21 Aufrufe, jedes Wort einmal benutzt: True, alle passen: True
raetsel3: 28 Aufrufe, jedes Wort einmal benutzt: True, alle passen: True
raetsel4: 57 Aufrufe, jedes Wort einmal benutzt: True, alle passen: True
None ([1, 0], 3)


Auch hier kommt eine gültige Zuordnung heraus. Da die Rätsel nur eine Lösung haben, ist es dieselbe wie beim Backtracking.

Weitere Gedanken:

- **Mehr als eine Lösung:** Sind zwei Wörter vertauschbar (gleiche Wörter wie `wurde`, `wurde`), ist das egal, der Text bleibt derselbe. Gäbe es echte Alternativen, würde das Programm nur die erste liefern.
- **Satzzeichen mitten im Wort oder am Anfang** (Anführungszeichen, Klammern, Bindestrich) gibt es in den Beispielen nicht. Mit `zerlege` ginge es ähnlich: auch vorne abschneiden.
- **MiniZinc:** Hier nicht aufgenommen. Das Problem ist ein reines Matching, das Backtracking und die verbessernden Wege schon vollständig und schnell lösen.